In [ ]:
# pip install requests json python-binance pandas openpyxl tqdm
# python = 3.12

In [ ]:
# 관련 docs
# [01]. binance api
#       https://developers.binance.com/docs/binance-spot-api-docs

In [10]:
import requests
import json
from binance.client import Client

import pandas as pd
import openpyxl

import os
import time

from tqdm import tqdm

In [11]:
# 주의
KEYS = pd.read_excel("private_keys.xlsx")
api_key = KEYS['API Key'][0]
api_secret = KEYS['Secret Key'][0]

client = Client(api_key, api_secret)


In [ ]:
KLINE_COLUMNS = [
    'Open time', 'Open', 'High', 'Low', 'Close', 'Volume', 
    'Close time', 'Quote asset volume', 'Number of trades', 
    'Taker buy base asset volume', 'Taker buy quote asset volume', 'Ignore'
]

class BinanceDataCollector:
    """
    바이낸스 API를 사용하여 심볼 정보 및 캔들스틱 데이터를 수집하는 클래스입니다.
    """
    def __init__(self, api_key, api_secret):
        # binance.client.Client 객체를 인스턴스 변수로 저장
        self.client = Client(api_key, api_secret)
        self.all_symbols = self.get_all_symbols()

    def get_all_symbols(self):
        """
        Binance 상장된 모든 Symbol 추출목적
        """
        info = self.client.get_exchange_info() # 심볼 + 필터
        
        symbols = [x['symbol'] for x in info['symbols']]
        print(f"전체 심볼 개수 : {len(symbols)}")
        return symbols

    def get_binance_klines(self, symbol, interval, start_str, end_str=None):
        # start_str과 end_str은 날짜 문자열 (예: '1 Jan, 2024').

        """
        Binance API에서 캔들스틱 데이터 추출
        , API 과다호출로 인한 ban을 받기 위해서 1회 호출시 symbol개수 한정
        """
        try:
            klines = self.client.get_historical_klines(
                symbol, 
                interval, 
                start_str, 
                end_str,
                limit = 500 # 한 번에 500개 symbol만 호출
            )
            if not klines:
                print(f"{symbol} : 심볼 데이터 없음. 이유는 체크필요.")
                return None
            
            # DataFrame으로 변환하고 컬럼 이름 명시적으로 설정
            data = pd.DataFrame(klines, columns=KLINE_COLUMNS)

            # 데이터 타입 변환 및 인덱스 설정 (기존 get_binance_klines 로직 통합)
            data['Open time'] = pd.to_datetime(data['Open time'], unit='ms')
            data['Close time'] = pd.to_datetime(data['Close time'], unit='ms')
            
            numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
            for col in numeric_cols:
                # 숫자형 컬럼을 float으로 변환
                data[col] = pd.to_numeric(data[col])
                
            data = data.set_index('Open time')
            
            print(f"{symbol} : ({len(data)}개) 데이터 추출 완료")
            return data
        
        except Exception as e:
            print(f"{symbol} : 데이터 추출 오류 코드({e})")
            return None
        finally:
            time.sleep(1) # ban 방지

In [25]:
data_collector = BinanceDataCollector(api_key, api_secret)

INTERVAL = Client.KLINE_INTERVAL_1HOUR
START_DATE = "1 May, 2024"

전체 심볼 개수 : 3399


In [26]:
# 데이터 dict형식으로 받을 예정
all_klines_data = {}

print("==========Data Extract Process is started.==========")

# 인스턴스 변수에 저장된 전체 심볼 목록 사용
all_symbols = data_collector.all_symbols

# test
# for symbol in tqdm(all_symbols,miniters=100, desc="처리 중"):
for symbol in tqdm(all_symbols[:5],miniters=100, desc="처리 중"):
    df_klines = data_collector.get_binance_klines(symbol, INTERVAL, START_DATE)
    
    if df_klines is not None:
        all_klines_data[symbol] = df_klines

print("==========Data Extract Process is done.==========")

==========Data Extract Process is started.==========


처리 중:   0%|          | 0/5 [00:00<?, ?it/s]

ETHBTC : (500개) 데이터 추출 완료
LTCBTC : (500개) 데이터 추출 완료
BNBBTC : (500개) 데이터 추출 완료
NEOBTC : (500개) 데이터 추출 완료
QTUMETH : (500개) 데이터 추출 완료


처리 중: 100%|██████████| 5/5 [00:05<00:00,  1.17s/it]

==========Data Extract Process is done.==========


In [27]:
df_klines

,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,Ignore
Open time,,,,,,,,,,,
2024-05-01 00:00:00,0.001220,0.001225,0.001181,0.001181,2278.3,2024-05-01 00:59:59.999,2.74272590,89,12.90000000,0.01543440,0
2024-05-01 01:00:00,0.001183,0.001188,0.001180,0.001188,126.7,2024-05-01 01:59:59.999,0.15035310,11,0.00000000,0.00000000,0
2024-05-01 02:00:00,0.001182,0.001182,0.001181,0.001181,110.5,2024-05-01 02:59:59.999,0.13059810,2,0.00000000,0.00000000,0
2024-05-01 03:00:00,0.001181,0.001181,0.001181,0.001181,0.0,2024-05-01 03:59:59.999,0.00000000,0,0.00000000,0.00000000,0
2024-05-01 04:00:00,0.001184,0.001184,0.001184,0.001184,98.5,2024-05-01 04:59:59.999,0.11662400,1,0.00000000,0.00000000,0
...,...,...,...,...,...,...,...,...,...,...,...
2024-05-21 15:00:00,0.001026,0.001026,0.001015,0.001015,120.5,2024-05-21 15:59:59.999,0.12308480,11,1.00000000,0.00101700,0
2024-05-21 16:00:00,0.001013,0.001048,0.001011,0.001036,2016.7,2024-05-21 16:59:59.999,2.07825980,59,1480.60000000,1.52572120,0
2024-05-21 17:00:00,0.001048,0.001059,0.001048,0.001055,401.4,2024-05-21 17:59:59.999,0.42337020,13,389.10000000,0.41040410,0
